# 16 — Ensemble v1: multi-scale TTA + greedy WBF over the earlier models

**Result:** holdout mAP 0.2627, public LB 0.2420 (outputs in `ensemble16/`).

**Starting point.** Best public score so far **0.21201** (15b: 3×RT-DETR-L@640 + final RetinaNet, WBF), organizer baseline **0.22163**. Holdout COCO mAP on the 951-frame / 7-video holdout has tracked the public score closely (0.203→0.2078, 0.209→0.2120, YOLO 0.184→0.1873), so it is used as the selection signal.

**What is new compared with 15b — each step is measured separately on the holdout:**
1. **More model families in the pool.** YOLO11s (the best single model, never fused before), Faster R-CNN (two-stage, a different error profile), YOLO CV folds and the new 1280-px YOLO from **16a** when its bundle is on Drive.
2. **Test-time augmentation per source.** YOLO at 960/1280/1536 px and horizontal flip; RT-DETR at 800/960 px instead of 640 (RT-DETR shrinks 1344×756 frames to 640×640, which hurts the ~24 % small objects); RetinaNet flip and 960 px.
3. **Greedy forward selection of families** on all 951 holdout frames (15b's 4-vs-3-video tune/check split disagreed with itself). Then one pass over family weights and fusion parameters. Every tried configuration is logged.
4. **Temporal rescoring** (bonus task): a detection's score is blended with the score of its best same-class match in the neighbouring frames, which suppresses flickering false positives.
5. Robustness table on the two holdout halves, and two submissions (with / without temporal) to pick the two final Kaggle entries.

**Run order:** top to bottom. GPU is only used in sections 3 and 8; everything else is CPU and cached on Drive (`MyDrive/UCU_AI_HW2/ensemble_cache`, outputs in `ensemble16/`). Rerunning skips everything already cached. Expected GPU time with all options on: ≈1–1.5 h holdout inference + ≈1–1.5 h test inference for the selected sources.

**Files expected under `MyDrive/UCU_AI_HW2/`** (missing optional ones are skipped automatically):
`visdrone_competition.zip`, `rtdetr_tuning_essentials.zip`, `retinanet_final_6ep/best.pt`, `retinanet_v2_tuning/resolution_768/fold_*/latest.pt`, `ensemble_cache/` (from 15b) · optional: `faster_rcnn_tuning/...`, `yolo_leaderboard_essentials.zip` (notebook 09, YOLO11s 18 ep), `yolo_tuning_results.zip` (notebook 05), `yolo_hr_bundle.zip` or `yolo_hr/` (notebook 16a).

In [16]:
%pip -q install "ultralytics==8.4.204" pycocotools faster-coco-eval

## 1 — Setup and options

In [17]:
import os, gc, io, re, sys, json, gzip, time, shutil, zipfile, hashlib, contextlib, itertools
import multiprocessing as mp
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, torch, cv2
from PIL import Image
from tqdm.auto import tqdm
from torchvision.transforms.functional import pil_to_tensor
from torchvision.models.detection import retinanet_resnet50_fpn_v2, fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import RTDETR, YOLO
from google.colab import drive

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/UCU_AI_HW2')
OUT = DRIVE / 'ensemble16'; OUT.mkdir(parents=True, exist_ok=True)
CACHE = DRIVE / 'ensemble_cache'; CACHE.mkdir(exist_ok=True)
LOCAL = Path('/content/ensemble16'); CKPTS = LOCAL / 'checkpoints'; CKPTS.mkdir(parents=True, exist_ok=True)
COMPETITION_ZIP = DRIVE / 'visdrone_competition.zip'
assert zipfile.is_zipfile(COMPETITION_ZIP), f'Missing {COMPETITION_ZIP}'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# ---------------- options ----------------
YOLO_SIZES   = [960, 1280, 1536]   # test-time scales for YOLO checkpoints trained at 960
YOLO_HR_SIZES = [1280, 1536]       # for the 16a model trained at 1280 (cached by 16a)
YOLO_FLIP    = True                # add a horizontally flipped pass at 1280
RT_PROBE_SIZES = [800, 960]        # RT-DETR fold 0 is probed at these sizes; the best is used for all folds
RETINA_TTA   = True                # final RetinaNet: hflip and 960-px passes
USE_FASTER   = True                # Faster R-CNN CV folds (if checkpoints exist)
USE_YOLO_CV  = True                # 6-epoch YOLO11s CV folds from 05 (if found)
MIN_GAIN     = 0.0010              # greedy selection stops when holdout mAP improves less than this
N_PROC       = max(1, min(4, os.cpu_count() or 1))   # parallel fusion workers (CPU); set 1 if a fusion cell hangs
OVERRIDE     = {}                  # e.g. {'yolo18': '/content/drive/MyDrive/.../best.pt'} if auto-discovery picks wrong file
print('Device', DEVICE, torch.cuda.get_device_name(0) if DEVICE != 'cpu' else '', '| CPU workers', N_PROC)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device cuda:0 Tesla T4 | CPU workers 2


## 2 — Checkpoints, annotations and frame order

Checkpoints are copied to local disk; cached predictions are tied to the **SHA-256 fingerprint of the exact weight file**, so a cache can never be reused for a different model.

In [18]:
def extract(zpath, member, dest):
    if not dest.is_file():
        with zipfile.ZipFile(zpath) as z, z.open(member) as s, open(dest, 'wb') as d:
            shutil.copyfileobj(s, d)
    return dest

# --- RT-DETR-L (08): lr2e4 folds; RetinaNet (07/12); Faster R-CNN (06) ---
WEIGHTS = {}   # label -> (kind, path)
with zipfile.ZipFile(DRIVE / 'rtdetr_tuning_essentials.zip') as z:
    names = z.namelist()
for k in (0, 1, 2):
    m = [n for n in names if re.search(fr'(^|/)lr2e4_fold_{k}/weights/best\.pt$', n)]
    assert len(m) == 1, m
    WEIGHTS[f'rt_f{k}'] = ('rt', extract(DRIVE / 'rtdetr_tuning_essentials.zip', m[0], CKPTS / f'rtdetr_lr2e4_fold_{k}.pt'))
if (DRIVE / 'retinanet_final_6ep/best.pt').is_file():
    WEIGHTS['retina_final'] = ('retina', DRIVE / 'retinanet_final_6ep/best.pt')
for k in (0, 1, 2):
    p = DRIVE / f'retinanet_v2_tuning/resolution_768/fold_{k}/latest.pt'
    if p.is_file(): WEIGHTS[f'retina_f{k}'] = ('retina', p)
FASTER = {0: DRIVE / 'faster_rcnn_tuning/resolution_768/latest.pt',
          1: DRIVE / 'faster_rcnn_tuning/resolution_768_3fold/fold_1/latest.pt',
          2: DRIVE / 'faster_rcnn_tuning/resolution_768_3fold/fold_2/latest.pt'}
if USE_FASTER and all(p.is_file() for p in FASTER.values()):
    WEIGHTS.update({f'faster_f{k}': ('faster', p) for k, p in FASTER.items()})

# --- YOLO checkpoints: discovered as loose files or inside ZIPs anywhere under UCU_AI_HW2 ---
YOLO_PATTERNS = [(r'(^|/)yolo11s_960_step3_18ep_adamw/weights/best\.pt$', 'yolo18'),        # notebook 09
                 (r'(^|/)yolo11s_960_step3_6ep_adamw_final/weights/best\.pt$', 'yolo6f'),   # notebook 11
                 (r'(^|/)yolo_hr/yolo_hr_final\.pt$', 'yolohr')]                            # notebook 16a
if USE_YOLO_CV:
    YOLO_PATTERNS += [(fr'(^|/)s960_adamw_lr0005_fold_{k}/weights/best\.pt$', f'yolocv_f{k}') for k in (0, 1, 2)]
found = defaultdict(list)
for f in sorted(DRIVE.rglob('*')):
    if CACHE in f.parents or f == COMPETITION_ZIP or not f.is_file():
        continue
    rel = str(f.relative_to(DRIVE))
    if f.suffix == '.pt':
        for pat, lab in YOLO_PATTERNS:
            if re.search(pat, rel): found[lab].append(('file', f, None))
    elif f.suffix == '.zip' and zipfile.is_zipfile(f):
        with zipfile.ZipFile(f) as z:
            for n in z.namelist():
                for pat, lab in YOLO_PATTERNS:
                    if re.search(pat, n): found[lab].append(('zip', f, n))
                if n.startswith('yolo_hr/cache/') and n.endswith('.json.gz') and not (CACHE / Path(n).name).exists():
                    extract(f, n, CACHE / Path(n).name)        # caches produced by 16a on Kaggle
for p in (DRIVE / 'yolo_hr' / 'cache').glob('*.json.gz'):          # caches produced by 16a on Colab
    if not (CACHE / p.name).exists(): shutil.copy2(p, CACHE / p.name)
for lab, hits in found.items():
    if lab in OVERRIDE: continue
    how, f, member = hits[0]
    if len(hits) > 1: print(f'NOTE {lab}: {len(hits)} candidates, using {f.name}:{member or ""}  (set OVERRIDE to change)')
    WEIGHTS[lab] = ('yolo', f if how == 'file' else extract(f, member, CKPTS / f'{lab}.pt'))
for lab, p in OVERRIDE.items():
    WEIGHTS[lab] = ('yolo', Path(p))
print(pd.DataFrame([(k, v[0], v[1].name, round(v[1].stat().st_size / 1e6, 1)) for k, v in WEIGHTS.items()],
                   columns=['label', 'kind', 'file', 'MB']).to_string(index=False))
if not any(v[0] == 'yolo' for v in WEIGHTS.values()):
    print('WARNING: no YOLO checkpoint found. Upload yolo_leaderboard_essentials.zip (Kaggle output of 09) and/or '
          'yolo_hr_bundle.zip (16a) to MyDrive/UCU_AI_HW2 - YOLO is the main addition of this notebook.')

       label   kind                   file    MB
       rt_f0     rt rtdetr_lr2e4_fold_0.pt  66.2
       rt_f1     rt rtdetr_lr2e4_fold_1.pt  66.2
       rt_f2     rt rtdetr_lr2e4_fold_2.pt  66.2
retina_final retina                best.pt 291.3
   retina_f0 retina              latest.pt 291.3
   retina_f1 retina              latest.pt 291.3
   retina_f2 retina              latest.pt 291.3
   faster_f0 faster              latest.pt 330.4
   faster_f1 faster              latest.pt 330.4
   faster_f2 faster              latest.pt 330.4
      yolo18   yolo              yolo18.pt  19.2
   yolocv_f0   yolo           yolocv_f0.pt  19.2
   yolocv_f1   yolo           yolocv_f1.pt  19.2
   yolocv_f2   yolo           yolocv_f2.pt  19.2


In [19]:
# ---- raw-prediction caches: {image_key: {'size':[w,h], 'detections':[[cls0..7, score, x1,y1,x2,y2], ...]}} ----
MIN_CONF, MAX_RAW_DETS = 0.001, 300
_FP = {}
def fingerprint(path):
    path = Path(path)
    if path not in _FP:
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            while block := f.read(8 * 1024 * 1024):
                h.update(block)
        _FP[path] = h.hexdigest()[:12]
    return _FP[path]

def _clean(boxes, classes, scores, w, h, flip):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4).copy()
    if flip:  # undo horizontal flip
        boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, w)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, h)
    classes = np.asarray(classes, dtype=int); scores = np.asarray(scores, dtype=float)
    ok = (classes >= 0) & (classes < 8) & (scores >= MIN_CONF) & (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
    order = np.argsort(-scores[ok], kind='stable')[:MAX_RAW_DETS]
    return [[int(c), float(s), *map(float, b)] for c, s, b in
            zip(classes[ok][order], scores[ok][order], boxes[ok][order])]

def predict_ultralytics(model, paths, imgsz, flip=False, batch=8, half=True, desc=''):
    """YOLO or RT-DETR (Ultralytics). flip=True runs on the mirrored frame and mirrors boxes back."""
    items, res = list(paths.items()), {}
    for s in tqdm(range(0, len(items), batch), desc=desc, leave=False):
        chunk = items[s:s + batch]
        if flip:
            src = [np.ascontiguousarray(cv2.imread(str(p))[:, ::-1]) for _, p in chunk]
        else:
            src = [str(p) for _, p in chunk]
        preds = model.predict(source=src, imgsz=imgsz, conf=MIN_CONF, iou=0.7, max_det=MAX_RAW_DETS,
                              half=half, device=0, batch=len(chunk), verbose=False, save=False, augment=False)
        assert len(preds) == len(chunk)
        for (key, _), r in zip(chunk, preds):
            h, w = r.orig_shape
            if r.boxes is None or len(r.boxes) == 0:
                dets = []
            else:
                dets = _clean(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(),
                              r.boxes.conf.cpu().numpy(), w, h, flip)
            res[str(key)] = {'size': [int(w), int(h)], 'detections': dets}
    return res

def save_cache(res, dest):
    tmp = Path(str(dest) + '.partial')
    with gzip.open(tmp, 'wt') as f:
        json.dump(res, f)
    tmp.replace(dest)

def load_cache(path):
    with gzip.open(path, 'rt') as f:
        return json.load(f)

with zipfile.ZipFile(COMPETITION_ZIP) as z:
    VAL_COCO = json.loads(z.read(next(n for n in z.namelist() if n.endswith('val/annotations_coco.json'))))
VAL_JSON = LOCAL / 'val_coco.json'; VAL_JSON.write_text(json.dumps(VAL_COCO))

DATA_ROOT = None
def ensure_data():
    """Extract the 11-GB competition ZIP to local disk only when some inference is not cached."""
    global DATA_ROOT
    if DATA_ROOT is None:
        base = Path('/content/visdrone'); base.mkdir(exist_ok=True)
        find = lambda: [p.parent.parent for p in base.rglob('train/annotations_coco.json')
                        if (p.parent.parent / 'test/images').is_dir()]
        if not find():
            print('Extracting competition data (one time, ~5 min)...')
            with zipfile.ZipFile(COMPETITION_ZIP) as z: z.extractall(base)
        DATA_ROOT = find()[0]
    return DATA_ROOT

# ---- holdout / test identifiers (identical to notebooks 14 and 15b) ----
VAL_STEP = 3
meta = pd.DataFrame([{'id': int(i['id']), 'file_name': i['file_name'],
                      'clip': Path(i['file_name']).parent.name} for i in VAL_COCO['images']])
meta = meta.sort_values(['clip', 'file_name']).reset_index(drop=True)
meta['n'] = meta.groupby('clip').cumcount()
val_table = meta.loc[meta['n'] % VAL_STEP == 0].copy()
VAL_IDS = val_table['id'].astype(int).tolist()
assert len(VAL_IDS) == 951, len(VAL_IDS)

def val_paths():
    root = ensure_data() / 'val'
    out = {}
    for row in val_table.itertuples():
        rel = Path(row.file_name)
        cands = [root / rel, root / 'images' / rel.parent.name / rel.name]
        p = next((c for c in cands if c.is_file()), None)
        assert p is not None, f'missing val image {rel}'
        out[str(int(row.id))] = p
    return out

def test_paths():
    root = ensure_data() / 'test' / 'images'
    files = [p for p in root.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    out = {f'{p.parent.name}_{p.stem}': p for p in files}
    assert len(out) == len(files) == 3811, len(out)
    return dict(sorted(out.items()))

CLIPS = sorted(val_table['clip'].unique())
HALF_A = val_table.loc[val_table['clip'].isin(CLIPS[:4]), 'id'].astype(str).tolist()
HALF_B = val_table.loc[val_table['clip'].isin(CLIPS[4:]), 'id'].astype(str).tolist()
VAL_KEYS = [str(i) for i in VAL_IDS]
VAL_SEQS = [g['id'].astype(str).tolist() for _, g in val_table.groupby('clip', sort=True)]

# Test keys look like clip_001_0000001; the holdout uses every 3rd frame, so we use the same
# temporal gap on the test videos (offset in frames = holdout gap / test gap).
_cached = sorted(CACHE.glob(f'test_all_rt_f0_{fingerprint(WEIGHTS["rt_f0"][1])}.json.gz'))
TEST_KEYS = sorted(json.loads(gzip.open(_cached[0], 'rt').read())) if _cached else list(test_paths())
assert len(TEST_KEYS) == 3811
test_df = pd.DataFrame({'key': TEST_KEYS})
test_df['clip'] = test_df.key.str.rsplit('_', n=1).str[0]
test_df['frame'] = test_df.key.str.rsplit('_', n=1).str[1].astype(int)
test_df = test_df.sort_values(['clip', 'frame'])
TEST_SEQS = [g['key'].tolist() for _, g in test_df.groupby('clip', sort=True)]
val_frame = val_table.assign(f=val_table.file_name.map(lambda s: int(Path(s).stem)))
VAL_GAP = float(val_frame.groupby('clip')['f'].diff().median())
TEST_GAP = float(test_df.groupby('clip')['frame'].diff().median())
TEST_OFFSET = max(1, int(round(VAL_GAP / TEST_GAP)))
print(f'holdout: {len(VAL_IDS)} frames, halves {len(HALF_A)}/{len(HALF_B)} | test: {len(TEST_KEYS)} frames in '
      f'{len(TEST_SEQS)} clips | frame gap val={VAL_GAP:g} test={TEST_GAP:g} -> test temporal offset {TEST_OFFSET}')

holdout: 951 frames, halves 471/480 | test: 3811 frames in 48 clips | frame gap val=3 test=1 -> test temporal offset 3


## 3 — Raw predictions per source (GPU, cached)

A **source** = checkpoint + inference setting (scale, flip). Labels without a suffix are exactly the settings of notebooks 14/15b, so their existing caches are reused.

In [20]:
def build_torchvision(kind, path, min_size):
    max_size = int(round(min_size * 5 / 3))       # 768 -> 1280 as in training
    if kind == 'retina':
        m = retinanet_resnet50_fpn_v2(weights=None, weights_backbone=None, min_size=min_size, max_size=max_size,
                                      score_thresh=MIN_CONF, nms_thresh=.5, detections_per_img=MAX_RAW_DETS,
                                      topk_candidates=1000)
        h = m.head.classification_head; n = m.anchor_generator.num_anchors_per_location()[0]
        h.cls_logits = torch.nn.Conv2d(h.cls_logits.in_channels, n * 8, 3, stride=1, padding=1); h.num_classes = 8
    else:
        m = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None, min_size=min_size, max_size=max_size,
                                    box_score_thresh=MIN_CONF, box_nms_thresh=.5, box_detections_per_img=MAX_RAW_DETS,
                                    rpn_pre_nms_top_n_test=1000, rpn_post_nms_top_n_test=500)
        m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 9)
    m.load_state_dict(torch.load(path, map_location='cpu', weights_only=True)['model'], strict=True)
    return m.eval().to(DEVICE)

def predict_torchvision(model, paths, flip, label_shift, desc=''):
    res = {}
    with torch.inference_mode():
        for key, p in tqdm(paths.items(), desc=desc, leave=False):
            with Image.open(p) as im:
                w, h = im.size
                x = pil_to_tensor(im.convert('RGB')).float().div_(255).to(DEVICE)
            if flip: x = x.flip(-1)
            d = model([x])[0]
            res[str(key)] = {'size': [w, h], 'detections': _clean(d['boxes'].cpu().numpy(),
                             d['labels'].cpu().numpy() + label_shift, d['scores'].cpu().numpy(), w, h, flip)}
    return res

DEFAULT_SIZE = {'rt': 640, 'retina': 768, 'faster': 768}
SPECS, FAMILY = {}, {}
def add_source(base, size=None, flip=False):
    """Register base checkpoint + inference setting; returns the source label."""
    kind, _ = WEIGHTS[base]
    size = size or DEFAULT_SIZE.get(kind)
    label = base + ('' if size == DEFAULT_SIZE.get(kind) else f'_s{size}') + ('_flip' if flip else '')
    # family = model without the fold index + inference setting, e.g. rt_f1_s800 -> rt_s800, retina_f2 -> retina_cv
    fam = re.sub(r'_f\d', '', base) + ('_cv' if re.match(r'(retina|faster)_f\d', base) else '') + label[len(base):]
    SPECS[label] = dict(base=base, kind=kind, size=size, flip=flip)
    FAMILY[label] = fam
    return label

PRED = {'val': {}, 'test': {}}
def get_source(label, split):
    """Load cached raw predictions (or run inference) and keep them as numpy arrays."""
    if label in PRED[split]:
        return PRED[split][label]
    sp = SPECS[label]; kind, ckpt = WEIGHTS[sp['base']]
    cname = 'val_step3' if split == 'val' else 'test_all'
    dest = CACHE / f'{cname}_{label}_{fingerprint(ckpt)}.json.gz'
    if not dest.is_file():
        assert DEVICE.startswith('cuda'), f'GPU needed to compute {dest.name}'
        paths = val_paths() if split == 'val' else test_paths()
        t = time.time()
        if kind in ('rt', 'yolo'):
            model = (RTDETR if kind == 'rt' else YOLO)(str(ckpt))
            res = predict_ultralytics(model, paths, sp['size'], sp['flip'],
                                      batch=2 if kind == 'rt' else (8 if sp['size'] <= 1280 else 4),
                                      half=(kind == 'yolo'), desc=f'{label} {split}')
        else:
            model = build_torchvision(kind, ckpt, sp['size'])
            res = predict_torchvision(model, paths, sp['flip'], 0 if kind == 'retina' else -1, desc=f'{label} {split}')
        del model; gc.collect(); torch.cuda.empty_cache()
        save_cache(res, dest)
        print(f'  computed {dest.name}: {(time.time() - t) * 1000 / len(paths):.0f} ms/img')
    raw = load_cache(dest)
    keys = VAL_KEYS if split == 'val' else TEST_KEYS
    assert set(raw) == set(keys), (label, split, len(raw))
    PRED[split][label] = {k: np.asarray(raw[k]['detections'], np.float32).reshape(-1, 6) for k in keys}
    return PRED[split][label]

In [21]:
import logging
from ultralytics.utils import LOGGER
class _HideHalfWarning(logging.Filter):
    def filter(self, record):
        return "'half' is deprecated" not in record.getMessage()
LOGGER.addFilter(_HideHalfWarning())

In [22]:
# ---- candidate pool (holdout only; test inference later runs only for the selected sources) ----
for b in [w for w in WEIGHTS if WEIGHTS[w][0] in ('rt', 'retina', 'faster')]:
    add_source(b)
if RETINA_TTA and 'retina_final' in WEIGHTS:
    add_source('retina_final', flip=True); add_source('retina_final', size=960)
for b in [w for w in WEIGHTS if WEIGHTS[w][0] == 'yolo']:
    sizes = YOLO_HR_SIZES if b == 'yolohr' else ([1280] if b.startswith('yolocv') else YOLO_SIZES)
    for s in sizes: add_source(b, size=s)
    if YOLO_FLIP and not b.startswith('yolocv'): add_source(b, size=1280, flip=True)
rt_probe = [add_source('rt_f0', size=s) for s in RT_PROBE_SIZES]

t0 = time.time()
for label in list(SPECS):
    get_source(label, 'val')
print(f'{len(SPECS)} holdout sources ready in {(time.time() - t0) / 60:.1f} min')

21 holdout sources ready in 0.6 min


## 4 — Fusion, evaluation, sanity check against 15b

The fusion rule is unchanged from 14/15b (class-aware weighted box fusion; each source contributes at most one box per cluster; score = weighted mean × coverage^p), only vectorised with NumPy (≈3× faster, identical output up to float32 rounding) and parallelised over images. Evaluation uses `faster-coco-eval` when it reproduces `pycocotools` on the sanity configuration, otherwise `pycocotools`.

In [23]:
import numpy as np
from collections import defaultdict

# ---------------- fusion (class-aware WBF, one box per source per cluster) ----------------
def fuse_arrays(arrays, weights, iou_thr=0.55, cov_pow=0.5, max_det=300, skip_thr=1e-3):
    """arrays: list of (N,6) float arrays [cls,score,x1,y1,x2,y2]; weights: list of floats.
    Same rule as notebooks 14/15b: boxes sorted by weight*score, each source joins a cluster at most once,
    box = (w*score)-weighted mean, score = mean_w(score) * (present_weight/total_weight)**cov_pow."""
    total = float(sum(weights))
    parts = []
    for si, (a, w) in enumerate(zip(arrays, weights)):
        if w <= 0 or a is None or len(a) == 0:
            continue
        a = a[a[:, 1] >= skip_thr]
        if len(a):
            parts.append(np.column_stack([a, np.full(len(a), si), np.full(len(a), w)]))
    if not parts:
        return np.zeros((0, 6), np.float32)
    B = np.concatenate(parts)
    out = []
    for c in np.unique(B[:, 0]):
        Bc = B[B[:, 0] == c]
        Bc = Bc[np.argsort(-(Bc[:, 1] * Bc[:, 7]), kind='stable')]
        n = len(Bc)
        boxes, scores = Bc[:, 2:6], Bc[:, 1]
        srcs, ws = Bc[:, 6].astype(np.int64), Bc[:, 7]
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        cbox = np.zeros((n, 4)); carea = np.zeros(n)
        swb = np.zeros((n, 4)); sws = np.zeros(n); sw = np.zeros(n)
        mask = np.zeros(n, np.int64); K = 0
        for i in range(n):
            b = boxes[i]; bit = np.int64(1) << srcs[i]; j = -1
            if K:
                cb = cbox[:K]
                iw = np.minimum(cb[:, 2], b[2]) - np.maximum(cb[:, 0], b[0])
                ih = np.minimum(cb[:, 3], b[3]) - np.maximum(cb[:, 1], b[1])
                inter = np.clip(iw, 0, None) * np.clip(ih, 0, None)
                iou = inter / (carea[:K] + area[i] - inter + 1e-9)
                iou[(mask[:K] & bit) != 0] = -1.0
                j = int(iou.argmax())
                if iou[j] <= iou_thr:
                    j = -1
            if j < 0:
                j = K; K += 1
            wsc = ws[i] * scores[i]
            swb[j] += wsc * b; sws[j] += wsc; sw[j] += ws[i]; mask[j] |= bit
            cbox[j] = swb[j] / sws[j]
            carea[j] = (cbox[j, 2] - cbox[j, 0]) * (cbox[j, 3] - cbox[j, 1])
        sc = np.minimum(1.0, (sws[:K] / sw[:K]) * (sw[:K] / total) ** cov_pow)
        keep = (sc >= skip_thr) & (cbox[:K, 2] > cbox[:K, 0]) & (cbox[:K, 3] > cbox[:K, 1])
        if keep.any():
            out.append(np.column_stack([np.full(keep.sum(), c), sc[keep], cbox[:K][keep]]))
    if not out:
        return np.zeros((0, 6), np.float32)
    R = np.concatenate(out)
    R = R[np.argsort(-R[:, 1], kind='stable')][:max_det]
    return R.astype(np.float32)


# ---------------- temporal rescoring ----------------
def pairwise_iou(a, b):
    iw = np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0])
    ih = np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1])
    inter = np.clip(iw, 0, None) * np.clip(ih, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)


def temporal_rescore(fused, sequences, offset=1, alpha=0.3, iou_thr=0.5):
    """fused: {image_id: (N,6)}; sequences: list of lists of image ids in temporal order (one list per clip).
    New score = (1-alpha)*s + alpha*support, support = mean over existing neighbours (t-offset, t+offset)
    of the best same-class neighbour score with IoU>=iou_thr (0 if no match)."""
    out = {}
    for seq in sequences:
        for t, img in enumerate(seq):
            D = fused[img]
            nb = [fused[seq[k]] for k in (t - offset, t + offset) if 0 <= k < len(seq)]
            if len(D) == 0 or not nb or alpha == 0:
                out[img] = D; continue
            support = np.zeros(len(D))
            for N in nb:
                if len(N) == 0:
                    continue
                for c in np.unique(D[:, 0]):
                    di = np.where(D[:, 0] == c)[0]; ni = np.where(N[:, 0] == c)[0]
                    if len(ni) == 0:
                        continue
                    iou = pairwise_iou(D[di, 2:6], N[ni, 2:6])
                    s = np.where(iou >= iou_thr, N[ni, 1][None, :], 0.0).max(1)
                    support[di] += s
            support /= len(nb)
            R = D.copy(); R[:, 1] = (1 - alpha) * D[:, 1] + alpha * support
            out[img] = R[np.argsort(-R[:, 1], kind='stable')]
    return out


_CTX = mp.get_context('fork')
def _fuse_chunk(args):
    keys, split, labels, weights, iou_thr, cov, max_det = args
    P = PRED[split]
    return {k: fuse_arrays([P[l][k] for l in labels], weights, iou_thr, cov, max_det) for k in keys}

def fuse_all(cfg, split='val'):
    labels = list(cfg['weights']); weights = [cfg['weights'][l] for l in labels]
    for l in labels: get_source(l, split)
    keys = VAL_KEYS if split == 'val' else TEST_KEYS
    args = lambda ks: (ks, split, labels, weights, cfg['iou'], cfg['cov'], cfg['max_det'])
    global N_PROC
    if N_PROC > 1:
        try:
            with _CTX.Pool(N_PROC) as pool:
                parts = pool.map(_fuse_chunk, [args(keys[i::N_PROC]) for i in range(N_PROC)])
            return {k: v for part in parts for k, v in part.items()}
        except Exception as e:
            print(f'parallel fusion unavailable ({type(e).__name__}); continuing on one core'); N_PROC = 1
    return _fuse_chunk(args(keys))

def apply_temporal(fused, cfg, split='val'):
    t = cfg.get('temporal')
    if not t: return fused
    seqs, off = (VAL_SEQS, 1) if split == 'val' else (TEST_SEQS, TEST_OFFSET)
    return temporal_rescore(fused, seqs, off, t['alpha'], t['iou'])

def _coco_dets(fused, keys):
    return [{'image_id': int(k), 'category_id': int(c) + 1, 'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': float(s)}
            for k in keys for c, s, x1, y1, x2, y2 in fused[k].tolist()]

with contextlib.redirect_stdout(io.StringIO()):
    GT_SLOW = COCO(str(VAL_JSON))
try:
    from faster_coco_eval import COCO as FCOCO, COCOeval_faster
    with contextlib.redirect_stdout(io.StringIO()):
        GT_FAST = FCOCO(str(VAL_JSON))
except Exception as e:
    print('faster-coco-eval unavailable:', e); GT_FAST = None
FAST_OK = False

def coco_eval(fused, keys=None, fast=None):
    keys = keys or VAL_KEYS
    fast = FAST_OK if fast is None else fast
    gt, Ev = (GT_FAST, COCOeval_faster) if fast else (GT_SLOW, COCOeval)
    with contextlib.redirect_stdout(io.StringIO()):
        e = Ev(gt, gt.loadRes(_coco_dets(fused, keys)), 'bbox')
        e.params.imgIds = [int(k) for k in keys]; e.params.catIds = list(range(1, 9)); e.params.maxDets = [1, 10, 100]
        e.evaluate(); e.accumulate(); e.summarize()
    s = e.stats
    return dict(mAP=float(s[0]), AP50=float(s[1]), AP75=float(s[2]), AP_small=float(s[3]),
                AP_medium=float(s[4]), AP_large=float(s[5]))

LOG = []      # every evaluated configuration (for the report)
_MEMO = {}
def score(cfg, stage, note=''):
    key = json.dumps(cfg, sort_keys=True)
    if key not in _MEMO:
        t = time.time()
        _MEMO[key] = coco_eval(apply_temporal(fuse_all(cfg), cfg))
        LOG.append({'stage': stage, 'note': note, **_MEMO[key], 'sec': round(time.time() - t, 1), 'config': key})
        pd.DataFrame(LOG).to_csv(OUT / 'search_log.csv', index=False)
    return _MEMO[key]['mAP']

# Sanity: 15b's selected configuration ("final_balanced") must give the 15b holdout number (0.20908).
_ret = {'retina_final': 1.0} if 'retina_final' in SPECS else {f'retina_f{k}': 1 / 3 for k in range(3)}
CFG_15B = dict(weights={**{f'rt_f{k}': 1 / 3 for k in range(3)}, **_ret}, iou=.55, cov=.5, max_det=300)
t = time.time(); f15 = fuse_all(CFG_15B); t_fuse = time.time() - t
t = time.time(); slow = coco_eval(f15, fast=False); t_slow = time.time() - t
if GT_FAST is not None:
    t = time.time(); fastm = coco_eval(f15, fast=True); t_fast = time.time() - t
    FAST_OK = abs(fastm['mAP'] - slow['mAP']) < 2e-4 and abs(fastm['AP_small'] - slow['AP_small']) < 2e-4
    print(f'faster-coco-eval mAP {fastm["mAP"]:.5f} vs pycocotools {slow["mAP"]:.5f} -> using fast: {FAST_OK} '
          f'({t_fast:.0f}s vs {t_slow:.0f}s)')
print(f'15b config on full holdout: mAP {slow["mAP"]:.5f} (15b reported 0.20908) | fusion {t_fuse:.0f}s')
LOG.append({'stage': '0_reference_15b', 'note': 'rt_f0-2 + retina_final, iou .55, cov .5', **slow, 'config': json.dumps(CFG_15B)})

faster-coco-eval mAP 0.20908 vs pycocotools 0.20908 -> using fast: True (7s vs 62s)
15b config on full holdout: mAP 0.20908 (15b reported 0.20908) | fusion 51s


In [24]:
# Resume: reuse the configuration selected before the disconnect (saved in section 7)
saved = json.loads((OUT / 'final_config.json').read_text())
CFG_FINAL, CFG_FUSED = saved['final'], saved['no_temporal']
for label in CFG_FINAL['weights']:          # register sources created in section 5 (e.g. rt_f1_s800)
    if label not in SPECS:
        m = re.match(r'(.+?)(?:_s(\d+))?(_flip)?$', label)
        add_source(m.group(1), size=int(m.group(2)) if m.group(2) else None, flip=bool(m.group(3)))
print('sources:', list(CFG_FINAL['weights']))
print({k: v for k, v in CFG_FINAL.items() if k != 'weights'})

sources: ['yolocv_f0_s1280', 'yolocv_f1_s1280', 'yolocv_f2_s1280', 'faster_f0', 'faster_f1', 'faster_f2', 'yolo18_s1280_flip', 'yolo18_s960', 'retina_final_s960', 'yolo18_s1536']
{'iou': 0.55, 'cov': 1.0, 'max_det': 500}


## 5 — Single sources and families

A **family** groups the CV folds of the same model at the same inference setting (e.g. `rt_s640` = 3 RT-DETR folds at 640 px, fused with equal weight). Selection works on families, which keeps the search small (less holdout overfitting) and treats fold-ensembles as one model.

In [11]:
rows = []
for label in SPECS:
    m = coco_eval({k: PRED['val'][label][k] for k in VAL_KEYS})
    rows.append({'source': label, 'family': FAMILY[label], **m})
single = pd.DataFrame(rows).sort_values('mAP', ascending=False)
single.to_csv(OUT / 'single_sources.csv', index=False)
display(single.round(4).head(40))

# RT-DETR resolution probe on fold 0: extend the best size to all folds if it beats 640 px.
rt640 = single.set_index('source').loc['rt_f0', 'mAP']
best_probe = max(rt_probe, key=lambda l: single.set_index('source').loc[l, 'mAP'])
gain = single.set_index('source').loc[best_probe, 'mAP'] - rt640
print(f'RT-DETR fold 0: 640px {rt640:.4f} | best probe {best_probe} {rt640 + gain:.4f} ({gain:+.4f})')
if gain > 0.002:
    size = SPECS[best_probe]['size']
    for k in (1, 2): get_source(add_source(f'rt_f{k}', size=size), 'val')
FAMILIES = defaultdict(list)
for label in SPECS:
    FAMILIES[FAMILY[label]].append(label)
FAMILIES = {f: m for f, m in FAMILIES.items()
            if not (f.startswith('rt_s') and len(m) < 3)}   # single-fold RT probes are not families
rows = []
for f, mem in FAMILIES.items():
    cfg = dict(weights={l: 1 / len(mem) for l in mem}, iou=.55, cov=.5, max_det=300)
    rows.append({'family': f, 'members': len(mem), 'mAP': score(cfg, '1_family', f)})
fam_table = pd.DataFrame(rows).sort_values('mAP', ascending=False)
display(fam_table.round(4))

,source,family,mAP,AP50,AP75,AP_small,AP_medium,AP_large
18,yolocv_f2_s1280,yolocv_s1280,0.2080,0.3867,0.1960,0.1029,0.1865,0.3428
15,yolo18_s1280_flip,yolo18_s1280_flip,0.1923,0.3650,0.1740,0.1097,0.1701,0.3140
13,yolo18_s1280,yolo18_s1280,0.1916,0.3726,0.1709,0.1003,0.1792,0.3037
12,yolo18_s960,yolo18_s960,0.1842,0.3504,0.1660,0.0738,0.1624,0.3172
14,yolo18_s1536,yolo18_s1536,0.1835,0.3611,0.1621,0.1066,0.1801,0.2827
11,retina_final_s960,retina_final_s960,0.1821,0.3571,0.1621,0.0674,0.1717,0.3290
8,faster_f1,faster_cv,0.1807,0.3783,0.1526,0.0804,0.1634,0.3434
9,faster_f2,faster_cv,0.1802,0.3841,0.1417,0.0663,0.1656,0.3400
7,faster_f0,faster_cv,0.1773,0.3708,0.1484,0.0514,0.1633,0.3426
3,retina_final,retina_final,0.1756,0.3380,0.1601,0.0580,0.1575,0.3335


RT-DETR fold 0: 640px 0.1537 | best probe rt_f0_s800 0.1502 (-0.0035)


,family,members,mAP
10,yolocv_s1280,3,0.2211
3,faster_cv,3,0.2027
9,yolo18_s1280_flip,1,0.1923
7,yolo18_s1280,1,0.1916
6,yolo18_s960,1,0.1842
8,yolo18_s1536,1,0.1835
5,retina_final_s960,1,0.1821
0,rt,3,0.1781
1,retina_final,1,0.1756
2,retina_cv,3,0.1700


## 6 — Greedy forward selection, then weights and fusion parameters

Start from the best family; at each step add the family that raises holdout mAP the most; stop when the gain is below `MIN_GAIN`. Then a single coordinate pass over family weight multipliers and the fusion IoU / coverage power / output cap. All configurations go to `search_log.csv`.

In [12]:
def make_cfg(fams, mult=None, iou=.55, cov=.5, max_det=300, temporal=None):
    w = {}
    for f in fams:
        for l in FAMILIES[f]:
            w[l] = (mult or {}).get(f, 1.0) / len(FAMILIES[f])
    cfg = dict(weights=w, iou=iou, cov=cov, max_det=max_det)
    if temporal: cfg['temporal'] = temporal
    return cfg

selected = [fam_table.iloc[0]['family']]
best = fam_table.iloc[0]['mAP']
print(f'start: {selected} {best:.4f}')
while len(selected) < 8:
    trials = {f: score(make_cfg(selected + [f]), '2_greedy', '+' + f) for f in FAMILIES if f not in selected}
    if not trials: break
    f, m = max(trials.items(), key=lambda kv: kv[1])
    print(f'  + {f:<22} -> {m:.4f} ({m - best:+.4f})')
    if m - best < MIN_GAIN: break
    selected.append(f); best = m
print('selected families:', selected, f'holdout mAP {best:.4f}')

# --- one coordinate pass over family weights ---
mult = {f: 1.0 for f in selected}
for f in selected:
    for v in (0.5, 0.75, 1.5, 2.0):
        trial = dict(mult, **{f: v})
        m = score(make_cfg(selected, trial), '3_weights', f'{f}x{v}')
        if m > best + 2e-4: best, mult = m, trial
print('weights:', mult, f'{best:.4f}')

# --- fusion parameters ---
P = dict(iou=.55, cov=.5, max_det=300)
for name, values in (('iou', (.5, .6, .65)), ('cov', (.25, .75, 1.0)), ('max_det', (500,))):
    for v in values:
        trial = dict(P, **{name: v})
        m = score(make_cfg(selected, mult, **trial), '4_fusion', f'{name}={v}')
        if m > best + 2e-4: best, P = m, trial
print('fusion params:', P, f'{best:.4f}')
CFG_FUSED = make_cfg(selected, mult, **P)

start: ['yolocv_s1280'] 0.2211
  + faster_cv              -> 0.2427 (+0.0216)
  + yolo18_s1280_flip      -> 0.2475 (+0.0048)
  + yolo18_s960            -> 0.2498 (+0.0023)
  + retina_final_s960      -> 0.2510 (+0.0012)
  + yolo18_s1536           -> 0.2522 (+0.0012)
  + rt                     -> 0.2527 (+0.0005)
selected families: ['yolocv_s1280', 'faster_cv', 'yolo18_s1280_flip', 'yolo18_s960', 'retina_final_s960', 'yolo18_s1536'] holdout mAP 0.2522
weights: {'yolocv_s1280': 2.0, 'faster_cv': 1.5, 'yolo18_s1280_flip': 1.5, 'yolo18_s960': 1.0, 'retina_final_s960': 1.0, 'yolo18_s1536': 1.0} 0.2559
fusion params: {'iou': 0.55, 'cov': 1.0, 'max_det': 500} 0.2627


## 7 — Temporal rescoring (bonus: data temporality)

For every fused detection we look for the best same-class box (IoU ≥ τ) in the previous and next sampled frame and set `score' = (1-α)·score + α·mean(neighbour scores)` (0 if unmatched). Objects that persist keep their score; one-frame false positives lose score. On the holdout the neighbours are 3 frames apart; on the test videos we use the same 3-frame gap (`TEST_OFFSET`), so the tuned α/τ transfer. No boxes are added or moved, only re-ranked.

In [13]:
best_t, t_best = best, None
for alpha, tau in itertools.product((0.2, 0.35, 0.5), (0.3, 0.5)):
    m = score(dict(CFG_FUSED, temporal={'alpha': alpha, 'iou': tau}), '5_temporal', f'alpha={alpha} iou={tau}')
    print(f'  alpha={alpha} iou={tau}: {m:.4f} ({m - best:+.4f})')
    if m > best_t + 5e-4: best_t, t_best = m, {'alpha': alpha, 'iou': tau}
CFG_FINAL = dict(CFG_FUSED, temporal=t_best) if t_best else CFG_FUSED
print('temporal:', t_best, f'-> holdout mAP {best_t:.4f}')

  alpha=0.2 iou=0.3: 0.2569 (-0.0058)
  alpha=0.2 iou=0.5: 0.2597 (-0.0030)
  alpha=0.35 iou=0.3: 0.2457 (-0.0170)
  alpha=0.35 iou=0.5: 0.2524 (-0.0103)
  alpha=0.5 iou=0.3: 0.2273 (-0.0354)
  alpha=0.5 iou=0.5: 0.2380 (-0.0247)
temporal: None -> holdout mAP 0.2627


### Robustness on the two holdout halves

The configuration was chosen on all 7 videos; a gain that appears on both halves (4 videos / 3 videos) is more likely to transfer to the hidden test set than one driven by a single video.

In [14]:
rows = []
for name, cfg in (('15b (public 0.21201)', CFG_15B), ('16b fused, no temporal', CFG_FUSED), ('16b final', CFG_FINAL)):
    fused = apply_temporal(fuse_all(cfg), cfg)
    for part, keys in (('all 7 videos', VAL_KEYS), ('half A (4 videos)', HALF_A), ('half B (3 videos)', HALF_B)):
        rows.append({'config': name, 'part': part, **coco_eval(fused, keys)})
robust = pd.DataFrame(rows)
robust.to_csv(OUT / 'robustness_halves.csv', index=False)
display(robust.pivot(index='config', columns='part', values='mAP').round(4))
display(robust[robust.part == 'all 7 videos'].drop(columns='part').round(4))
(OUT / 'final_config.json').write_text(json.dumps({'final': CFG_FINAL, 'no_temporal': CFG_FUSED,
                                                     'test_temporal_offset': TEST_OFFSET}, indent=2))

part,all 7 videos,half A (4 videos),half B (3 videos)
config,,,
15b (public 0.21201),0.2091,0.2560,0.1969
16b final,0.2627,0.3038,0.2357
"16b fused, no temporal",0.2627,0.3038,0.2357


,config,mAP,AP50,AP75,AP_small,AP_medium,AP_large
0,15b (public 0.21201),0.2091,0.4087,0.1861,0.0923,0.1922,0.3585
3,"16b fused, no temporal",0.2627,0.4846,0.2464,0.1285,0.2330,0.4377
6,16b final,0.2627,0.4846,0.2464,0.1285,0.2330,0.4377


871

## 8 — Test inference for the selected sources and submissions

Only the sources in the final configuration are run on the 3,811 test frames (cached on Drive, so a disconnect loses at most one source). Two files are written: `submission_16b_final.csv` (with temporal rescoring) and `submission_16b_no_temporal.csv`, a natural pair for the two final Kaggle selections.

In [27]:
for label in CFG_FINAL['weights']:
    get_source(label, 'test')

def write_submission(cfg, path):
    fused = apply_temporal(fuse_all(cfg, 'test'), cfg, 'test')
    rows = []
    for k in TEST_KEYS:
        tok = []
        for c, s, x1, y1, x2, y2 in fused[k].tolist():
            tok += [str(int(c)), f'{s:.6f}', f'{x1:.2f}', f'{y1:.2f}', f'{x2 - x1:.2f}', f'{y2 - y1:.2f}']
        rows.append({'image_id': k, 'PredictionString': ' '.join(tok)})
    sub = pd.DataFrame(rows)
    assert len(sub) == 3811 and sub.image_id.is_unique
    assert all(len(x.split()) % 6 == 0 for x in sub.PredictionString)
    sub.to_csv(path, index=False)
    n = sub.PredictionString.str.split().str.len().sum() // 6
    print(f'{path.name}: {n / 3811:.0f} boxes/frame, {path.stat().st_size / 1e6:.0f} MB')

write_submission(CFG_FINAL, OUT / 'submission_16b_final.csv')
if CFG_FINAL != CFG_FUSED:
    write_submission(CFG_FUSED, OUT / 'submission_16b_no_temporal.csv')

submission_16b_final.csv: 490 boxes/frame, 70 MB


## 9 — Summary for the report

In [28]:
fam_of = lambda cfg: sorted({FAMILY[l] for l in cfg['weights']})
summary = pd.DataFrame([
    {'step': '15b reference', 'families': fam_of(CFG_15B), 'holdout_mAP': slow['mAP'], 'public_LB': 0.21201},
    {'step': 'best single family', 'families': [fam_table.iloc[0]['family']], 'holdout_mAP': fam_table.iloc[0]['mAP']},
    {'step': '+ greedy families', 'families': selected,
     'holdout_mAP': score(make_cfg(selected), '2_greedy', 'final greedy')},
    {'step': '+ weights & fusion params', 'families': selected, 'holdout_mAP': score(CFG_FUSED, '4_fusion', 'final')},
    {'step': '+ temporal rescoring', 'families': selected, 'holdout_mAP': score(CFG_FINAL, '5_temporal', 'final')},
])
summary.to_csv(OUT / 'summary_steps.csv', index=False)
display(summary.round(4))
print('Selected config:', json.dumps({k: v for k, v in CFG_FINAL.items() if k != 'weights'}),
      '| sources:', list(CFG_FINAL['weights']))
print('Files in', OUT, ':', sorted(p.name for p in OUT.iterdir()))

Selected config: {"iou": 0.55, "cov": 1.0, "max_det": 500} | sources: ['yolocv_f0_s1280', 'yolocv_f1_s1280', 'yolocv_f2_s1280', 'faster_f0', 'faster_f1', 'faster_f2', 'yolo18_s1280_flip', 'yolo18_s960', 'retina_final_s960', 'yolo18_s1536']
Files in /content/drive/MyDrive/UCU_AI_HW2/ensemble16 : ['final_config.json', 'robustness_halves.csv', 'search_log.csv', 'single_sources.csv', 'submission_16b_final.csv', 'summary_steps.csv']


,step,families,holdout_mAP,public_LB
0,15b reference,"[retina_final, rt]",0.2091,0.212
1,best single family,[yolocv_s1280],0.2211,NaN
2,+ greedy families,"[yolocv_s1280, faster_cv, yolo18_s1280_flip, y...",0.2522,NaN
3,+ weights & fusion params,"[yolocv_s1280, faster_cv, yolo18_s1280_flip, y...",0.2627,NaN
4,+ temporal rescoring,"[yolocv_s1280, faster_cv, yolo18_s1280_flip, y...",0.2627,NaN


Selected config: {"iou": 0.55, "cov": 1.0, "max_det": 500} | sources: ['yolocv_f0_s1280', 'yolocv_f1_s1280', 'yolocv_f2_s1280', 'faster_f0', 'faster_f1', 'faster_f2', 'yolo18_s1280_flip', 'yolo18_s960', 'retina_final_s960', 'yolo18_s1536']
Files in /content/drive/MyDrive/UCU_AI_HW2/ensemble16 : ['final_config.json', 'robustness_halves.csv', 'search_log.csv', 'single_sources.csv', 'submission_16b_final.csv', 'summary_steps.csv']


**Notes for the defense / report**
- Holdout = 7 original validation videos, never trained on by any model (the RetinaNet/YOLO "final" models used them only for validation curves; 16a exports the last epoch). Fusion choices were made on these videos, so the final holdout number is optimistic by the amount of selection; the halves table and the public LB are the checks.
- `search_log.csv` lists every configuration tried, including the ones that did not help (e.g. families the greedy step rejected, weights and fusion parameters that were reverted, temporal settings).
- Families can only be compared fairly within this notebook: all sources are evaluated with the same COCOeval, same 951 frames, same score floor (0.001) and per-image cap (300).